### Install Automatic1111 Web UI
- First, clone the official Web UI repository into Kaggle’s working directory.

In [1]:
!git clone -b dev https://github.com/AUTOMATIC1111/stable-diffusion-webui /kaggle/working/stable-diffusion-webui
%cd /kaggle/working/stable-diffusion-webui

Cloning into '/kaggle/working/stable-diffusion-webui'...
remote: Enumerating objects: 34994, done.
remote: Total 34994 (delta 0), reused 0 (delta 0), pack-reused 34994 (from 1)
Receiving objects: 100% (34994/34994), 35.67 MiB | 31.43 MiB/s, done.
Resolving deltas: 100% (24367/24367), done.
/kaggle/working/stable-diffusion-webui


### Download Stable Diffusion 1.5 Base Model
- Fetch the v1.5 checkpoint and a VAE (Variational Autoencoder) for better image quality.

In [2]:
!mkdir -p /kaggle/temp/models
!wget -O models/Stable-diffusion/v1-5-pruned-emaonly.safetensors "https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"
# !wget -O models/Stable-diffusion/dreamshaper_8.safetensors \
# "https://civitai.com/api/download/models/128713?type=Model&format=SafeTensor&size=pruned&fp=fp16"
!wget -O models/VAE/vae-ft-mse-840000-ema-pruned.safetensors "https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors"

--2026-04-26 11:35:40--  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
Resolving huggingface.co (huggingface.co)... 13.226.251.20, 13.226.251.81, 13.226.251.112, ...
Connecting to huggingface.co (huggingface.co)|13.226.251.20|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors [following]
--2026-04-26 11:35:40--  https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/66d19580e2632490a6bc5829/2ac63bfb6186057d88b65c3aa47ec90fd8d9aa3269164fc37fed1cb6f1a1efd0?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Content-Sha256=UNSIGNED-PAYLOAD&X-Amz-Credential=cas%2F20260426%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Da

### Install Dependencies
- Uninstall any pre-installed Torch versions and install the exact ones needed for our GPU setup, along with Web UI requirements and ngrok.


In [3]:
# !pip uninstall -y torch torchvision torchaudio xformers peft accelerate diffusers transformers
# !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
# !pip install xformers==0.0.29.post3 --index-url https://download.pytorch.org/whl/cu124
# !pip install -r /kaggle/working/stable-diffusion-webui/requirements_versions.txt
# !pip install -qq pyngrok

# Install uv (fast Python package manager)
!pip install -q uv

# Uninstall existing conflicting packages
!uv pip uninstall  torch torchvision torchaudio xformers peft accelerate diffusers transformers

# Install PyTorch with CUDA 12.4
!uv pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu124

# Install xformers
!uv pip install -q xformers==0.0.29.post3 \
    --index-url https://download.pytorch.org/whl/cu124

# Install Stable Diffusion WebUI requirements
!uv pip install -r /kaggle/working/stable-diffusion-webui/requirements_versions.txt

# Install ngrok helper
!uv pip install -q pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 3.6 MB/s eta 0:00:00
Using Python 3.11.13 environment at: /usr
Uninstalled 7 packages in 2.83s
 - accelerate==1.9.0
 - diffusers==0.34.0
 - peft==0.16.0
 - torch==2.6.0+cu124 (from https://download.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp311-cp311-linux_x86_64.whl)
 - torchaudio==2.6.0+cu124 (from https://download.pytorch.org/whl/cu124/torchaudio-2.6.0%2Bcu124-cp311-cp311-linux_x86_64.whl)
 - torchvision==0.21.0+cu124 (from https://download.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp311-cp311-linux_x86_64.whl)
 - transformers==4.53.3
Using Python 3.11.13 environment at: /usr
Resolved 132 packages in 1.65s
   Building filterpy==1.4.5
   Building filterpy==1.4.5
   Building jsonmerge==1.8.0
   Building filterpy==1.4.5
   Building jsonmerge==1.8.0
⠙ Preparing packages... (0/49)
   Building filterpy==1.4.5
   Building jsonmerge==1.8.0
⠙ Preparing packages... (0/49)
   Building filterpy==1.4.5
   Building jsonmerge==1.8.0


### Restart Session
- Restarting ensures Kaggle reloads the correct GPU libraries before we launch the Web UI (Important for GPU initialization).


### Configure ngrok
- Ngrok creates a secure URL so we can access the Web UI running in Kaggle from our browser

In [4]:
from pyngrok import ngrok

# Set your ngrok auth token (get it from ngrok.com)
ngrok.set_auth_token("3CtRVhR3XWdCro0Fw1kBQG8t9yf_58QcKTWVxp8kYe4yWRniu")

### Move into the directory
- As we restarted the session, we need to move to the web-ui folder to start the UI


In [5]:
%cd /kaggle/working/stable-diffusion-webui

/kaggle/working/stable-diffusion-webui


### Launch the Web UI
- Finally, we run the launch command to start Stable Diffusion Web UI and open it through ngrok

In [6]:
import threading
import time
import subprocess
import requests # New import
from pyngrok import ngrok

def kill_existing_tunnels():
    """Kill all existing ngrok tunnels to free up slots."""
    try:
        ngrok.disconnect_all()
        print("All existing ngrok tunnels disconnected.")
    except Exception as e:
        print(f"Error disconnecting tunnels: {e}")

def run_webui():
    """Function to run the Automatic1111 WebUI."""
    # Using --no-share to avoid Gradio's own public URL, as we are using ngrok.
    # The --listen flag is crucial for allowing external connections via ngrok.
    subprocess.run([
        "python", "launch.py",
        "--listen",
        "--port", "7860",
        "--enable-insecure-extension-access",
        "--no-download-sd-model",
        "--xformers",
        "--skip-torch-cuda-test",
        "--clip-models-path", "/kaggle/working/clip-vit-large-patch14"
    ])

def wait_for_server(port, timeout=120):
    """Waits for the server to start listening on the specified port."""
    start_time = time.time()
    while True:
        if time.time() - start_time > timeout:
            raise TimeoutError("Server did not start within the specified timeout.")
        try:
            requests.get(f"http://localhost:{port}")
            print(f"Server is up and running on port {port}.")
            return True
        except requests.exceptions.ConnectionError:
            print("Waiting for WebUI server to start...")
            time.sleep(20)

# --- Main execution flow ---

# 1. Clean up existing tunnels to avoid conflicts.
print("Cleaning up existing ngrok tunnels...")
kill_existing_tunnels()

# 2. Start the WebUI in a background thread.
print("Starting Automatic1111 WebUI...")
webui_thread = threading.Thread(target=run_webui, daemon=True)
webui_thread.start()

# 3. Wait for the WebUI server to become available.
try:
    wait_for_server(7860)
except TimeoutError as e:
    print(f"Error: {e}")
    # You may want to stop the script here if the server fails to start.
    exit(1)

# 4. Create and display the ngrok tunnel.
print("Creating ngrok tunnel...")
try:
    public_url = ngrok.connect(7860)
    print(f"\n🎉 Stable Diffusion WebUI is accessible at:")
    print(f"🔗 {public_url}")
    print("\nClick the link above to access the WebUI interface!")

    # Keep the main thread alive so the WebUI thread continues to run
    # until the notebook cell execution is stopped.
    webui_thread.join()

except Exception as e:
    print(f"Error creating tunnel: {e}")
    print("Check your ngrok authentication token or try restarting the session.")

Cleaning up existing ngrok tunnels...
Error disconnecting tunnels: module 'pyngrok.ngrok' has no attribute 'disconnect_all'
Starting Automatic1111 WebUI...
Waiting for WebUI server to start...


Cloning into '/kaggle/working/stable-diffusion-webui/repositories/stable-diffusion-webui-assets'...
Cloning into '/kaggle/working/stable-diffusion-webui/repositories/stable-diffusion-stability-ai'...
Cloning into '/kaggle/working/stable-diffusion-webui/repositories/generative-models'...
Cloning into '/kaggle/working/stable-diffusion-webui/repositories/k-diffusion'...
Cloning into '/kaggle/working/stable-diffusion-webui/repositories/BLIP'...


Waiting for WebUI server to start...
Waiting for WebUI server to start...


2026-04-26 11:37:53.304795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777203473.723403     190 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777203473.847483     190 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Waiting for WebUI server to start...


/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Python 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
Version: v1.10.1-96-g1937682a
Commit hash: 1937682a20f7f0442311a1ede68f9f0cb480163b
Installing clip
Cloning assets into /kaggle/working/stable-diffusion-webui/repositories/stable-diffusion-webui-assets...
Cloning Stable Diffusion into /kaggle/working/stable-diffusion-webui/repositories/stable-diffusion-stability-ai...
Cloning Stable Diffusion XL into /kaggle/working/stable-diffusion-webui/repositories/generative-models...
Cloning K-diffusion into /kaggle/working/stable-diffusion-webui/repositories/k-diffusion...
Cloning BLIP into /kaggle/working/stable-diffusion-webui/repositories/BLIP...
Launching Web UI with arguments: --listen --port 7860 --enable-insecure-extension-access --no-download-sd-model --xformers --skip-torch-cuda-test --clip-models-path /kaggle/working/clip-vit-large-patch14


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Server is up and running on port 7860.
Creating ngrok tunnel...
Error creating tunnel: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://countless-irrigate-vitality.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}

Check your ngrok authentication token or try restarting the session.
